# Lecture 04 — Imbalanced learning: rare events and decisions

A synthetic transaction example separates ranking, probability scale, and threshold policy.

![Teaching figure for imbalanced learning](../../docs/figures/04_imbalanced_learning.svg)

*Counts computed from the complete supplied fraud dataset, before splitting. The minority count explains why accuracy alone is insufficient.*

## Read counts before reading a metric

Suppose a detector has recall (TPR) 80% and false-positive rate 1%. Among 10,000 transactions with 1% fraud, expect 80 true alerts and 99 false alerts: precision is about 45%. At 0.1% fraud, expect 8 true alerts and about 100 false alerts: precision is about 7%. This assumes the same conditional detector behavior at both prevalences; changing population can change those rates too.

An always-negative rule can have high accuracy while missing every fraud. Its cost is `number_of_frauds * cost_per_missed_fraud`, even though it creates zero alerts. Compare this cost with the chosen policy, not just with accuracy. A threshold is an action rule chosen on validation rows; a class weight changes the fitted objective and can change the interpretation of raw probabilities.

## 1. Why accuracy can mislead

At positive prevalence $\pi$, the always-negative rule has accuracy $1-\pi$ and recall zero. For 3% prevalence it scores 97% accuracy while finding nothing. Report class counts and error types. At threshold $t$, predict $\hat y=\mathbf1[p\ge t]$; precision is $TP/(TP+FP)$ and recall is $TP/(TP+FN)$. Moving $t$ trades false alarms and missed events.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    auc,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    brier_score_loss,
)
from xgboost import XGBClassifier


def repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "data" / "fraud.csv").exists():
            return path
    raise FileNotFoundError("Could not find data/fraud.csv")


ROOT = repo_root()
data = pd.read_csv(ROOT / "data" / "fraud.csv")
features = [
    "amount",
    "hour",
    "card_present",
    "international",
    "account_age_days",
    "transactions_24h",
    "merchant_risk",
]
X, y = data[features], data["fraud"].astype(int)
print(f"rows={len(data)}, positives={y.sum()}, prevalence={y.mean():.3%}")
# Split first. The test rows remain untouched until all choices are frozen.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=14, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=15, stratify=y_dev
)
print("sizes:", len(X_train), len(X_valid), len(X_test))


def empirical_cost(actual, scores, threshold, c_fn=20.0, c_fp=1.0):
    predicted = scores >= threshold
    false_negative = np.sum((actual.to_numpy() == 1) & ~predicted)
    false_positive = np.sum((actual.to_numpy() == 0) & predicted)
    return c_fn * false_negative + c_fp * false_positive

## 2. Logistic risk and class weights

For $p=\sigma(\eta)$ and $\eta=x^T\beta$, Bernoulli negative log likelihood is $\ell=\log(1+e^\eta)-y\eta$. Its gradient is $(p-y)x$; Hessian is $p(1-p)xx^T\succeq0$. An L2 penalty adds curvature and shrinks slopes.

Weighting multiplies loss by $w_y$. Under ideal constant class weights, the weighted optimum satisfies $q/(1-q)=(w_1/w_0)p/(1-p)$, so weighting shifts posterior odds and probability scale. It may also change finite-sample ranking, but it does not guarantee calibrated natural probabilities.

## 2a. Why class weights shift the posterior

At fixed features $x$, let the natural positive probability be $p$. The conditional weighted Bernoulli risk for a proposed probability $q$ is $R(q|x)=-w_1p\log q-w_0(1-p)\log(1-q)$. Its derivative is $-w_1p/q+w_0(1-p)/(1-q)$. Setting this to zero gives $q/(1-q)=(w_1/w_0)p/(1-p)$. Thus weighting shifts odds even with an ideal learner; fitted weighted probabilities should not automatically be read as natural risk.

In [ ]:
negative, positive = np.bincount(y_train, minlength=2)
ratio = negative / positive
models = {
    "Logistic unweighted": make_pipeline(
        StandardScaler(), LogisticRegression(C=1, max_iter=2000, random_state=14)
    ),
    "Logistic balanced": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1, class_weight="balanced", max_iter=2000, random_state=14
        ),
    ),
    "XGBoost weighted": XGBClassifier(
        n_estimators=180,
        max_depth=3,
        learning_rate=0.06,
        min_child_weight=4,
        subsample=0.85,
        colsample_bytree=0.9,
        reg_lambda=2,
        scale_pos_weight=ratio,
        eval_metric="logloss",
        n_jobs=1,
        random_state=14,
    ),
}
for model in models.values():
    model.fit(X_train, y_train)
valid_scores = {
    name: model.predict_proba(X_valid)[:, 1] for name, model in models.items()
}
rows = []
for name, scores in valid_scores.items():
    precision, recall, _ = precision_recall_curve(y_valid, scores)
    rows.append(
        {
            "model": name,
            "AP": average_precision_score(y_valid, scores),
            "trapezoid_PR_area": auc(recall[::-1], precision[::-1]),
            "ROC_AUC": roc_auc_score(y_valid, scores),
            "Brier": brier_score_loss(y_valid, scores),
        }
    )
pd.DataFrame(rows).set_index("model").round(4)

## 3. PR and ROC metrics

No-skill expected precision is prevalence. Average precision weights precision by recall increments; trapezoid area linearly interpolates curve points and generally differs. ROC-AUC summarizes ranking over false-positive rates, but high ROC-AUC does not guarantee useful alert precision at rare prevalence. None chooses a threshold.

For recall levels ordered from strict to loose thresholds, $\operatorname{AP}=\sum_j(R_j-R_{j-1})P_j$. This weights each attained precision by the additional recall. The prevalence line is a population no-skill precision reference; finite-sample average precision for random rankings need not equal it exactly.

## Precision depends on prevalence

At a fixed threshold, let TPR be recall and FPR be the false-positive rate. Bayes' rule gives

$$\operatorname{Precision}=\frac{\pi\,\operatorname{TPR}}{\pi\,\operatorname{TPR}+(1-\pi)\,\operatorname{FPR}}.$$

When prevalence $\pi$ is small, even a modest FPR can create many false alerts. This identity explains why ROC ranking can look strong while precision at a useful recall remains limited.

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, scores in valid_scores.items():
    precision, recall, _ = precision_recall_curve(y_valid, scores)
    axes[0].plot(
        recall,
        precision,
        label=f"{name} AP={average_precision_score(y_valid, scores):.3f}",
    )
    fpr, tpr, _ = roc_curve(y_valid, scores)
    axes[1].plot(fpr, tpr, label=f"{name} AUC={roc_auc_score(y_valid, scores):.3f}")
axes[0].axhline(y_valid.mean(), color="black", linestyle="--", label="prevalence")
axes[0].set(xlabel="recall", ylabel="precision", title="Validation PR")
axes[1].plot([0, 1], [0, 1], color="black", linestyle="--")
axes[1].set(
    xlabel="false positive rate", ylabel="true positive rate", title="Validation ROC"
)
for axis in axes:
    axis.legend(fontsize=8)
plt.tight_layout()

## 4. Validation cost and final test

With calibrated natural probability $p$ and constant costs, predict positive when $C_{FP}(1-p)\le C_{FN}p$, giving $p\ge C_{FP}/(C_{FP}+C_{FN})$. This assumes calibrated risk and unconstrained actions. We choose empirical thresholds on validation scores using stated costs, freeze them, and evaluate test once.

In [ ]:
C_FN, C_FP = 20.0, 1.0
grid = np.unique(np.r_[0.0, np.linspace(0.01, 0.99, 197), 1.0])
thresholds = {}
for name, scores in valid_scores.items():
    costs = [empirical_cost(y_valid, scores, t, C_FN, C_FP) for t in grid]
    thresholds[name] = float(grid[int(np.argmin(costs))])
pd.DataFrame(
    {
        "threshold": thresholds,
        "validation_cost": {
            n: empirical_cost(y_valid, valid_scores[n], t, C_FN, C_FP)
            for n, t in thresholds.items()
        },
        "alert_rate": {n: np.mean(valid_scores[n] >= t) for n, t in thresholds.items()},
    }
).T

### Inspect the validation tradeoff before freezing the policy

The first minimum returned by `np.argmin` wins; because thresholds are sorted ascending, equal-cost ties choose the smallest threshold. This tends to send more alerts among tied policies. Costs are illustrative units with false-negative cost 20 and false-positive cost 1. Both panels use validation rows only.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, scores in valid_scores.items():
    costs = [empirical_cost(y_valid, scores, t, C_FN, C_FP) for t in grid]
    alerts = [(scores >= t).sum() for t in grid]
    axes[0].plot(grid, costs, label=name)
    axes[1].plot(grid, alerts, label=name)
    axes[0].scatter(thresholds[name], min(costs), s=30)
axes[0].set(xlabel="Threshold", ylabel="Validation cost", title="Choose a policy using validation")
axes[1].set(xlabel="Threshold", ylabel="Number of validation alerts", title="Operational workload")
for axis in axes:
    axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Final-test reference policy

The always-negative policy has no alerts and misses every positive. Its accuracy is one minus test prevalence. Compare the frozen models below against this reference; neither the reference nor the model test results are used to tune a threshold.

In [ ]:
test_negative, test_positive = np.bincount(y_test, minlength=2)
baseline = pd.DataFrame(
    [
        {
            "accuracy": test_negative / len(y_test),
            "recall": 0.0,
            "alerts": 0,
            "missed positives": test_positive,
        }
    ],
    index=["always negative"],
)
baseline

In [ ]:
rows = []
for name, model in models.items():
    scores = model.predict_proba(X_test)[:, 1]
    prediction = scores >= thresholds[name]
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    rows.append(
        {
            "model": name,
            "threshold": thresholds[name],
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "precision": precision_score(y_test, prediction, zero_division=0),
            "recall": recall_score(y_test, prediction, zero_division=0),
            "AP": average_precision_score(y_test, scores),
            "ROC_AUC": roc_auc_score(y_test, scores),
            "cost": C_FN * fn + C_FP * fp,
            "test_alert_rate": prediction.mean(),
        }
    )
rows.append({
    "model": "Always negative", "threshold": np.nan,
    "TP": 0, "FP": 0, "FN": int(test_positive), "TN": int(test_negative),
    "precision": 0.0, "recall": 0.0, "AP": np.nan, "ROC_AUC": np.nan,
    "cost": C_FN * test_positive, "test_alert_rate": 0.0,
})
# AP and ROC-AUC are omitted for this action-only reference policy.
pd.DataFrame(rows).set_index("model").round(4)

## Limits

Threshold search can overfit validation. Weighted outputs need not be calibrated to natural prevalence. No resampling is performed, and splitting precedes all fitting. This controlled synthetic data cannot support operational claims.